# Importing libraries

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

from datasets import load_dataset

from sklearn.metrics import precision_score, recall_score, f1_score, hamming_loss
from memory_profiler import memory_usage
from time import perf_counter

# Importing dataset

In [2]:
ds = load_dataset("higopires/RePro-categories-multilabel")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,review_text,ENTREGA,OUTROS,PRODUTO,CONDICOESDERECEBIMENTO,INADEQUADA,ANUNCIO
0,"Aparelho muito bom, confiável e com valor aqui...",0,0,1,0,0,0
1,"A história é muito boa, porém o autor ""enrolou...",0,0,1,0,0,0
2,"Entrega rápida, produto muito bom Amei. Pratic...",1,0,1,0,0,0
3,Produto otimo so falta o carregador da maquina...,0,0,1,1,0,0
4,a proteção anti queda não é boa se cair de fr...,0,0,1,0,0,0
...,...,...,...,...,...,...,...
7997,amei o produto. chegou no prazo e em perfeito ...,1,0,1,1,0,0
7998,Ótima embalagem. Produto entregue no prazo. Re...,1,0,1,1,0,0
7999,"ótimo produto, super recomendo .,Entrega bem r...",1,0,1,0,0,0
8000,"Veio tudo certinho, dentro do prazo e o produt...",1,0,1,1,0,0


# Dataset preprocessing

In [3]:
train_df = train_df.rename(columns={'review_text': 'text'})
val_df = val_df.rename(columns={'review_text': 'text'})
test_df = test_df.rename(columns={'review_text': 'text'})

In [4]:
class TextDataset(Dataset):
    def __init__(self, texts, label_matrix, tokenizer, max_len):
        """
        texts: a pandas Series or list of strings
        label_matrix: a pandas DataFrame or 2D NumPy array of shape [num_samples, num_labels]
                      Each row i has the 0/1 labels for text i.
        tokenizer: a transformers tokenizer
        max_len: maximum sequence length
        """
        self.texts = texts.tolist()
        # Convert the label matrix into a NumPy array if it isn't already
        self.labels = label_matrix.values if hasattr(label_matrix, 'values') else label_matrix
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        labels = self.labels[idx]  # shape: [num_labels]

        # Tokenize
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            # Convert labels to float so it works with BCEWithLogitsLoss
            'labels': torch.tensor(labels, dtype=torch.float)
        }


In [5]:
MAX_LEN = 128
BATCH_SIZE = 32

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

label_cols = [col for col in train_df.columns if col != 'text']

# Create datasets
train_dataset = TextDataset(
    texts=train_df['text'],
    label_matrix=train_df[label_cols],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df['text'],
    label_matrix=val_df[label_cols],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df['text'],
    label_matrix=test_df[label_cols],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Neural network class (LSTM)

In [6]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(LSTMClassifier, self).__init__()

        # For multi-label, output_dim = number_of_labels
        self.embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout
        )

        # If bidirectional=True, final hidden state has 2*hidden_dim
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        outputs, (hidden, cell) = self.lstm(embedded)

        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        hidden = self.dropout(hidden)
        logits = self.fc(hidden)  # shape [batch_size, output_dim]

        return logits  # raw logits for each label


# Instancing the LSTM model, criterion and optimizer

In [7]:
embedding_dim = 128
hidden_dim = 128
output_dim = len(label_cols)
n_layers = 2
bidirectional = True
dropout = 0.3

model = LSTMClassifier(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')
model = model.to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Using cuda device


# Training and evaluation functions

In [9]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    losses = []
    correct_predictions = 0
    total_labels = 0

    all_labels = []
    all_preds = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)  # shape: (batch_size, num_labels)

        optimizer.zero_grad()

        # Forward pass -> logits: [batch_size, num_labels]
        logits = model(input_ids)

        # Compute loss
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        # Convert logits to predictions in {0,1}
        preds = (torch.sigmoid(logits) > 0.5).float()

        # Count how many individual labels are predicted correctly
        correct_predictions += (preds == labels).sum().item()
        total_labels += labels.numel()

        # Store for metric calculation
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    # Calculate mean loss
    avg_loss = sum(losses) / len(losses)
    # Label-level accuracy
    accuracy = correct_predictions / total_labels

    # Convert to NumPy
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    # Macro-average precision, recall, F1
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    # Per-class F1
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

    # Hamming loss
    ham_loss = hamming_loss(all_labels, all_preds)

    return accuracy, avg_loss, precision, recall, f1_macro, f1_per_class, ham_loss


def eval_model(model, data_loader, criterion, device):
    model.eval()
    losses = []
    correct_predictions = 0
    total_labels = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)
            losses.append(loss.item())

            preds = (torch.sigmoid(logits) > 0.5).float()

            correct_predictions += (preds == labels).sum().item()
            total_labels += labels.numel()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    avg_loss = sum(losses) / len(losses)
    accuracy = correct_predictions / total_labels

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)
    ham_loss = hamming_loss(all_labels, all_preds)

    return accuracy, avg_loss, precision, recall, f1_macro, f1_per_class, ham_loss


# Training loop

In [10]:
def training_loop(epochs):
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        
        (
            train_acc, 
            train_loss, 
            train_prec, 
            train_rec, 
            train_f1_macro, 
            train_f1_per_class,
            train_ham_loss
        ) = train_epoch(model, train_loader, optimizer, criterion, device)
        
        (
            val_acc, 
            val_loss, 
            val_prec, 
            val_rec, 
            val_f1_macro, 
            val_f1_per_class,
            val_ham_loss
        ) = eval_model(model, val_loader, criterion, device)
        
        print(f"Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, "
              f"Precision(macro): {train_prec:.4f}, Recall(macro): {train_rec:.4f}, "
              f"F1(macro): {train_f1_macro:.4f}, Hamming: {train_ham_loss:.4f}")
        print(f"F1 Per Class (Train): {train_f1_per_class}")
        
        print(f"Val   Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, "
              f"Precision(macro): {val_prec:.4f}, Recall(macro): {val_rec:.4f}, "
              f"F1(macro): {val_f1_macro:.4f}, Hamming: {val_ham_loss:.4f}")
        print(f"F1 Per Class (Val):   {val_f1_per_class}")
        print("--------------------------------------------------")
    
    return (
        train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class, train_ham_loss,
        val_acc,   val_loss,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class,   val_ham_loss
    )


In [11]:

seeds = [2, 3, 5]
EPOCHS = 5

# Update your results DataFrame with Hamming Loss columns
results = pd.DataFrame(columns=[
    'seed',
    'train_loss', 'train_acc', 'train_prec', 'train_rec', 'train_f1', 'train_f1_per_class', 'train_ham',
    'val_loss',   'val_acc',   'val_prec',   'val_rec',   'val_f1',   'val_f1_per_class',   'val_ham',
    'test_loss',  'test_acc',  'test_prec',  'test_rec',  'test_f1',  'test_f1_per_class',  'test_ham',
    'max_memory_usage_train', 'max_vram_usage_train', 'total_time_train',
    'max_memory_usage_test',  'max_vram_usage_test',  'total_time_test'
])

for seed in seeds:
    torch.manual_seed(seed)
    
    # Reset / re-initialize model for each seed
    model = LSTMClassifier(
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=len(label_cols),  # Number of labels for multi-label
        n_layers=n_layers,
        bidirectional=bidirectional,
        dropout=dropout
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    # For multi-label classification, use BCEWithLogitsLoss
    criterion = nn.BCEWithLogitsLoss().to(device)
    
    # Reset CUDA memory tracking if using GPU
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    # -------- TRAINING -----------
    start_time_train = perf_counter()
    # training_loop should return:
    # (train_acc, train_loss, train_prec, train_rec, train_f1, train_f1_per_class, train_ham,
    #  val_acc,   val_loss,   val_prec,   val_rec,   val_f1,   val_f1_per_class,   val_ham)
    max_memory_usage_train, retval = memory_usage(
        (training_loop, (EPOCHS,), {}),
        retval=True, 
        max_usage=True
    )
    total_time_train = perf_counter() - start_time_train

    max_vram_usage_train = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if torch.cuda.is_available() else None
    )

    (
        train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class, train_ham,
        val_acc,   val_loss,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class,   val_ham
    ) = retval

    # Reset CUDA memory tracking before test evaluation
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    # -------- TESTING -----------
    start_time_test = perf_counter()
    # eval_model should return:
    # (test_acc, test_loss, test_prec, test_rec, test_f1, test_f1_per_class, test_ham)
    max_memory_usage_test, retval = memory_usage(
        (eval_model, (model, test_loader, criterion, device), {}),
        retval=True, 
        max_usage=True
    )
    total_time_test = perf_counter() - start_time_test

    max_vram_usage_test = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if torch.cuda.is_available() else None
    )

    test_acc, test_loss, test_prec, test_rec, test_f1, test_f1_per_class, test_ham = retval

    # -------- LOGGING -----------
    new_row = pd.DataFrame([[
        seed,
        train_loss, train_acc, train_prec, train_rec, train_f1_macro, train_f1_per_class, train_ham,
        val_loss,   val_acc,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class,   val_ham,
        test_loss,  test_acc,  test_prec,  test_rec,  test_f1,        test_f1_per_class,  test_ham,
        max_memory_usage_train, max_vram_usage_train, total_time_train,
        max_memory_usage_test,  max_vram_usage_test,  total_time_test
    ]], columns=results.columns)

    results = pd.concat([results, new_row], ignore_index=True)

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.3764, Accuracy: 0.8437, Precision(macro): 0.4433, Recall(macro): 0.2772, F1(macro): 0.3011, Hamming: 0.1563
F1 Per Class (Train): [0.60702875 0.2364532  0.88872032 0.06039808 0.01104972 0.00273224]
Val   Loss: 0.3220, Accuracy: 0.8784, Precision(macro): 0.5037, Recall(macro): 0.3771, F1(macro): 0.4068, Hamming: 0.1216
F1 Per Class (Val):   [0.83601286 0.43548387 0.92025316 0.24880383 0.         0.        ]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.2918, Accuracy: 0.8843, Precision(macro): 0.6299, Recall(macro): 0.4127, F1(macro): 0.4450, Hamming: 0.1157
F1 Per Class (Train): [0.85829633 0.5        0.92157485 0.35420011 0.03571429 0.        ]
Val   Loss: 0.2930, Accuracy: 0.8967, Precision(macro): 0.7127, Recall(macro): 0.4291, F1(macro): 0.4749, Hamming: 0.1033
F1 Per Class (Val):   [0.87419355 0.54497354 0.93765903 0.4017094  0.09090909 0.        ]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.2760, 

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
C:\Users\Rafael\AppData\Local\Temp\ipykernel_13884\3466194580.py:89: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_row], ignore_index=True)
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.3813, Accuracy: 0.8432, Precision(macro): 0.4866, Recall(macro): 0.2647, F1(macro): 0.3039, Hamming: 0.1568
F1 Per Class (Train): [0.52146597 0.20599739 0.88872112 0.18461538 0.01685393 0.00548697]
Val   Loss: 0.3190, Accuracy: 0.8815, Precision(macro): 0.5067, Recall(macro): 0.4208, F1(macro): 0.4522, Hamming: 0.1185
F1 Per Class (Val):   [0.86970684 0.492891   0.90803109 0.44268775 0.         0.        ]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.2859, Accuracy: 0.8875, Precision(macro): 0.6176, Recall(macro): 0.4410, F1(macro): 0.4778, Hamming: 0.1125
F1 Per Class (Train): [0.84353183 0.50448777 0.91947314 0.54106729 0.05797101 0.        ]
Val   Loss: 0.2931, Accuracy: 0.8939, Precision(macro): 0.6080, Recall(macro): 0.4803, F1(macro): 0.5067, Hamming: 0.1061
F1 Per Class (Val):   [0.88123924 0.57906459 0.92167102 0.6130031  0.04545455 0.        ]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.2513, 

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.3726, Accuracy: 0.8486, Precision(macro): 0.4479, Recall(macro): 0.2812, F1(macro): 0.3090, Hamming: 0.1514
F1 Per Class (Train): [0.60834327 0.27990431 0.89534709 0.06508475 0.         0.00551724]
Val   Loss: 0.3293, Accuracy: 0.8721, Precision(macro): 0.4927, Recall(macro): 0.3760, F1(macro): 0.4125, Hamming: 0.1279
F1 Per Class (Val):   [0.81099656 0.44444444 0.9086262  0.31111111 0.         0.        ]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.2931, Accuracy: 0.8829, Precision(macro): 0.6757, Recall(macro): 0.4000, F1(macro): 0.4341, Hamming: 0.1171
F1 Per Class (Train): [0.85173887 0.45899514 0.91993374 0.36177106 0.01212121 0.        ]
Val   Loss: 0.3004, Accuracy: 0.8860, Precision(macro): 0.5208, Recall(macro): 0.4161, F1(macro): 0.4407, Hamming: 0.1140
F1 Per Class (Val):   [0.84766214 0.38650307 0.92769608 0.48221344 0.         0.        ]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.3021, 

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [12]:
results.to_csv('results/lstm_multilabel1.csv', index=False)
results.head()

,seed,train_loss,train_acc,train_prec,train_rec,train_f1,train_f1_per_class,train_ham,val_loss,val_acc,...,test_rec,test_f1,test_f1_per_class,test_ham,max_memory_usage_train,max_vram_usage_train,total_time_train,max_memory_usage_test,max_vram_usage_test,total_time_test
0,2,0.236339,0.909773,0.776596,0.499099,0.534928,"[0.903478081909858, 0.6369646450129347, 0.9423...",0.090227,0.263364,0.909960,...,0.503165,0.519151,"[0.8910256410256411, 0.5955734406438632, 0.933...",0.103112,1205.437500,231.648926,17.751628,1205.511719,194.176270,0.765890
1,3,0.195820,0.925581,0.812394,0.627568,0.678237,"[0.929024340355901, 0.7042640990371389, 0.9410...",0.074419,0.230699,0.928236,...,0.609790,0.661627,"[0.9154228855721394, 0.6824034334763949, 0.936...",0.078616,1205.632812,232.119629,17.622878,1204.050781,195.025391,0.810671
2,5,0.261495,0.897213,0.765622,0.455886,0.498700,"[0.8927089508002372, 0.5674761758376883, 0.932...",0.102787,0.277889,0.899061,...,0.483147,0.506631,"[0.8881789137380192, 0.5454545454545454, 0.929...",0.108242,1205.449219,231.318359,18.097757,1204.000000,194.361328,0.773238


In [13]:
torch.save(model.state_dict(), 'results/lstm_multilabel1.pth')